# 経済データ分析Ⅱ　第8回
# プロビット

第7回の線形確率モデルでは持ち家の確率を年齢と年収の1次式で表したので、64歳で年収20百万円の世帯の予測確率が1.209になった。今回は確率がいつも0と1の間に入るように、確率を曲線で表す**プロビット**を学ぶ。確率を表す曲線とその背後にある潜在変数の考え方、推定と係数の読み方、予測確率の順に進める。

## Notebook の使い方

コードセルを上から順に選び、`Shift + Enter` で実行します。途中のセルから実行して「変数が見つからない」というエラー（`NameError`）が出たら、先頭のセルから実行し直してください。

JupyterLite では **Python (Pyodide)**、ローカルの Jupyter では Python カーネルを使います。データは Notebook の中で作るので、CSV ファイルの準備は要りません。

## 1. 準備

次のセルで、今回使うライブラリを読み込む。第7回と同じものである。`japanize_matplotlib_jlite` は図の文字を日本語で表示するためのパッケージ、`smf` は回帰式を文字で書いて推定する機能、`stats` は確率分布の値を計算する機能である。

In [ ]:
import sys
if sys.platform == 'emscripten':
    import pyodide_js
    await pyodide_js.loadPackage(['numpy', 'pandas', 'matplotlib', 'statsmodels', 'scipy'])
    import piplite
    await piplite.install('japanize-matplotlib-jlite')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib_jlite
import statsmodels.formula.api as smf
from scipy import stats

## 2. 第7回のデータ

第7回と同じ500世帯のデータを作り直す。シードも同じ202611なので、第7回と同じ値になる。各行の働きは第7回で説明したとおりである。持ち家の真の確率は

$$p_i=\Phi(-2.3+0.04\times年齢_i+0.09\times年収_i)$$

で、年収の単位は百万円である。

In [ ]:
rng = np.random.default_rng(202611)
n = 500
age = rng.integers(25, 65, size=n)
income = 7 * rng.lognormal(mean=-0.08, sigma=0.4, size=n)
p = stats.norm.cdf(-2.3 + 0.04 * age + 0.09 * income)
owned = (rng.random(n) < p).astype(int)
df = pd.DataFrame({'age': age, 'income': income, 'p': p, 'owned': owned})
df.head().round(3)

第7回の線形確率モデルも推定し直しておく。5節で予測確率を比べるときに使う。

In [ ]:
lpm = smf.ols('owned ~ age + income', data=df).fit()
print(lpm.params.round(4))

係数は第7回と同じで、年齢が0.0156、年収が0.0299である。

## 3. 確率を表す曲線

プロビットでは持ち家の確率を

$$P(y=1\mid x)=\Phi(\beta_0+\beta_1\,年齢+\beta_2\,年収)$$

と表す。括弧の中は第7回の線形確率モデルと同じ1次式で、それを標準正規分布の累積分布関数 $\Phi$ に入れる。第7回の2.2節で説明したように、$\Phi(z)$ は `stats.norm.cdf()` で計算でき、どんな $z$ を入れても0と1の間に入る。そのためプロビットの予測確率もいつも0と1の間に入る。

### 3.1. 1次式の値と持ち家の割合

第7回のデータで括弧の中の1次式と確率の関係を確かめる。真の係数で1次式の値を計算して `z` の列に入れ、先頭の5世帯の年齢、年収、`z`、確率 `p` を表示する。

In [ ]:
df['z'] = -2.3 + 0.04 * df['age'] + 0.09 * df['income']
df[['age', 'income', 'z', 'p']].head().round(3)

1行目の世帯は32歳で年収10.4百万円、1次式の値は −0.083 で、確率は0.467である。1次式の値は負にもなるが、$\Phi$ に入れた確率は0と1の間に入る。1〜4行目の世帯では1次式の値が0より小さく、確率も0.5より小さい。5行目の世帯は1次式の値が0.250で、確率は0.599になる。第7回の2.2節で $\Phi(0)=0.5$ を求めたので、1次式の値が0の世帯の確率はちょうど0.5である。

同じ節の累積分布関数の図では、$\Phi(z)$ の曲線はS字の形をしていた。$z=0$ の近くでは急に上がり、0や1に近いところでは平らになる。線形確率モデルの直線はどこでも同じ傾きだった。

1次式の値が近い世帯をまとめると、持ち家の割合がこの曲線に沿うかを確かめられる。第7回の2.4節で年齢を4つの区分に分けたのと同じように、1次式の値 `z` を5つの区分に分ける。`pd.cut()` と `.groupby()` は第7回と同じく数値を区分に分け、区分ごとに平均を求める。区分ごとに `z` の平均、持ち家の割合（`owned` の平均）、真の確率 `p` の平均を並べる。

In [ ]:
df['z_group'] = pd.cut(df['z'], bins=[-1.5, -0.5, 0, 0.5, 1, 2])
share = df.groupby('z_group', observed=True)[['z', 'owned', 'p']].mean()
share.round(3)

区分の名前の `(-0.5, 0.0]` は、−0.5より大きく0以下の範囲を表す。1次式の値が大きい区分ほど、持ち家の割合は高い。左の区分から順に0.219、0.392、0.534、0.798、0.833で、真の確率の平均0.250、0.412、0.591、0.760、0.885に近い。

各世帯の `owned`、$\Phi(z)$ の曲線、区分ごとの持ち家の割合を1つの図に描く。

- `z_line` は曲線を描くために−3から3までに並べた200個の点である。データの範囲より広くとって、S字の全体を見る。
- 区分ごとの持ち家の割合は赤い点で描く。点の横の位置は区分の `z` の平均、高さは持ち家の割合である。
- `ylim=(-0.05, 1.35)` は縦軸の範囲で、凡例が点に重ならないように上を空けている。凡例は左上に置く。

In [ ]:
z_line = np.linspace(-3, 3, 200)
fig, ax = plt.subplots()
ax.scatter(df['z'], df['owned'], s=10, alpha=0.3, label='各世帯の持ち家（1）かどうか')
ax.plot(z_line, stats.norm.cdf(z_line), label='Φ(z)')
ax.scatter(share['z'], share['owned'], color='red', label='区分ごとの持ち家の割合')
ax.set(xlabel='括弧の中の1次式 z', ylabel='持ち家の確率', ylim=(-0.05, 1.35))
ax.legend(loc='upper left')
plt.show()

赤い点は $\Phi$ の曲線の近くに並ぶ。1次式の値が近い世帯をまとめると、持ち家の割合はその値での $\Phi(z)$ に近い。プロビットはこの関係を式で表したモデルである。

各世帯を表す青い点は横軸の−1から2の間にあり、多くは−1から1の間にある。この範囲では $\Phi$ の曲線は直線に近い。第7回の3.3節の図で線形確率モデルの直線が真の確率の曲線とよく似ていたのは、そのためである。曲線が平らになって0や1に近づくのは、1次式の値がこの範囲より小さいところと大きいところである。

### 3.2. 潜在変数による見方

確率を表す曲線に $\Phi$ を使う理由を、持ち家を選ぶかどうかの決まり方から考える。データには現れないが、モデルの中で結果を決める変数を**潜在変数**という。ここでは、持ち家に住む良さが借家に住む良さをどれだけ上回るかを世帯ごとに表す潜在変数 $y^*$ があり、

$$y_i^*=-2.3+0.04\times年齢_i+0.09\times年収_i+\varepsilon_i$$

で決まるとする。$\varepsilon_i$ は第3回の誤差と同じく、式に入れていない要因（好み、貯蓄、家族の事情など）による違いを表し、標準正規分布に従う。$y_i^*$ が0より大きい世帯は持ち家（$y_i=1$）、0以下の世帯は持ち家でない（$y_i=0$）とする。データに現れるのは持ち家かどうかだけで、$y^*$ の値は観測されない。

括弧の中の1次式を $z_i$ と書くと $y_i^*=z_i+\varepsilon_i$ なので、持ち家になるのは $\varepsilon_i>-z_i$ のときである。5行目の世帯は $z=0.250$ なので、$\varepsilon$ が −0.250 より大きければ持ち家になる。この確率を第7回の2.2節と同じく密度関数の下の面積で見る。

- `z5` は5行目の世帯の1次式の値で、`df['z'][4]` で取り出す（行の番号は0から数える）。
- `eps_grid` は横軸の点、`eps_right` は塗る範囲の −0.250 から4までの点である。
- `ylim=(0, 0.5)` で縦軸の上を空け、凡例が曲線の山に重ならないようにする。凡例は左上に置く。

In [ ]:
z5 = df['z'][4]
eps_grid = np.linspace(-4, 4, 200)
eps_right = np.linspace(-z5, 4, 100)
fig, ax = plt.subplots()
ax.plot(eps_grid, stats.norm.pdf(eps_grid))
ax.fill_between(eps_right, stats.norm.pdf(eps_right), alpha=0.3, label='ε が −0.250 より大きくなる確率')
ax.set(xlabel='ε', ylabel='密度関数の高さ', ylim=(0, 0.5))
ax.legend(loc='upper left')
plt.show()

塗った部分の面積が、5行目の世帯が持ち家になる確率である。密度関数の曲線は0を中心に左右対称なので、−0.250より右の面積は0.250より左の面積 $\Phi(0.250)$ と同じになる。2つの値を計算して確かめる。

In [ ]:
print(f'−0.250より右の面積：{1 - stats.norm.cdf(-z5):.3f}')
print(f'Φ(0.250)：{stats.norm.cdf(z5):.3f}')

2つの値は0.599で一致し、3.1節の表の5行目の `p` とも同じである。どの世帯でも持ち家になる確率は

$$P(y_i=1)=P(\varepsilon_i>-z_i)=\Phi(z_i)$$

になる。式に入れていない要因 $\varepsilon$ が標準正規分布に従うと考えると、確率を表す曲線が $\Phi$ になる。

第7回では持ち家かどうかを `rng.random(n) < p` で決めた。潜在変数で決めても持ち家になる確率は同じ $p_i=\Phi(z_i)$ なので、2つの決め方は同じモデルを表す。真の確率はプロビットの形で作ったので、4節では推定値を真の係数 $-2.3$、$0.04$、$0.09$ と比べられる。

## 4. プロビットの推定

プロビットの係数は**最尤法**という方法で推定する。最尤法はデータの各世帯で観測された結果（持ち家かそうでないか）が最も起こりやすくなる係数を選ぶ方法で、statsmodels が係数を少しずつ変える計算を繰り返して探す。

`smf.probit('owned ~ age + income', data=df)` は、`smf.ols` と同じ書き方でプロビットの式を作る。`.fit(disp=0)` で推定する。`disp=0` は繰り返しの計算の途中経過を表示しない指定である。

In [ ]:
probit = smf.probit('owned ~ age + income', data=df).fit(disp=0)
pd.DataFrame({'推定値': probit.params, '標準誤差': probit.bse,
              '真の値': [-2.3, 0.04, 0.09]}).round(4)

推定値は定数が −2.4818、年齢が0.0431、年収が0.0874で、真の値 −2.3、0.04、0.09 に近い。

第4回の第4節では誤差を1,000回作り直し、推定値が標本ごとに変わって真の値の周りにばらつくことを見た。プロビットでも別の500世帯のデータなら推定値は変わる。標準誤差はこのばらつきの大きさを1つのデータから見積もった値で、年齢の係数では0.0055、年収の係数では0.0222である。

プロビットの係数は括弧の中の1次式がどれだけ変わるかを表す。年収が同じで年齢が1歳高い世帯では、1次式の値が0.0431大きい。年齢と年収の係数はどちらも正なので、年齢が高いほど、年収が多いほど1次式の値は大きくなり、持ち家の確率も高くなる。

確率が上がる幅は世帯によって違う。3.1節の図で見たように、$\Phi$ の曲線は $z=0$ の近くで急に上がり、0や1に近いところでは平らになる。1次式の値が同じ0.0431だけ大きくなっても、曲線の急なところにいる世帯では確率が大きく上がり、平らなところにいる世帯ではわずかしか上がらない。年齢が1歳高いと確率がどれだけ上がるかは、第9回で計算する。

## 5. 予測確率

プロビットでも予測確率は `.predict()` で計算する。プロビットの予測確率は推定した係数の1次式を $\Phi$ に入れた値である。第7回と同じ64歳で年収20百万円の世帯について、まず式で計算する。

$$\hat{p}=\Phi(\hat\beta_0+\hat\beta_1\times64+\hat\beta_2\times20)$$

係数は `probit.params['age']` のように名前で取り出せる。

In [ ]:
bp = probit.params
z64 = bp['Intercept'] + bp['age'] * 64 + bp['income'] * 20
print(f'括弧の中：{z64:.3f}')
print(f'式で計算した予測確率：{stats.norm.cdf(z64):.3f}')

括弧の中の1次式は2.023で1を超えるが、$\Phi$ に入れた値は0.978で1を超えない。

第7回と同じ2つの世帯（25歳で年収1百万円、64歳で年収20百万円）の表を作り、`probit.predict()` で予測する。

In [ ]:
new = pd.DataFrame({'age': [25, 64], 'income': [1, 20]})
probit.predict(new).round(3)

2つ目の世帯の値は式で計算した0.978と同じである。

線形確率モデルとプロビットの予測確率と、真の確率を並べる。

In [ ]:
pd.DataFrame({'線形確率モデル': lpm.predict(new), 'プロビット': probit.predict(new),
              '真の確率': stats.norm.cdf(-2.3 + 0.04 * new['age'] + 0.09 * new['income'])}).round(3)

64歳で年収20百万円の世帯の予測確率は、線形確率モデルでは1.209だったが、プロビットでは0.978で1を超えない。真の確率0.980にも近い。25歳で年収1百万円の世帯では、線形確率モデルが0.032、プロビットが0.094、真の確率が0.113である。

64歳の世帯について年収を0から40百万円まで動かし、3つの確率を図に描く。その前にデータの年収の最小と最大を確かめる。

In [ ]:
print(f"データの年収：{df['income'].min():.1f}〜{df['income'].max():.1f} 百万円")

データの年収は1.4〜20.2百万円なので、図の右側はデータの範囲の外になる。

年齢をすべて64歳にし、年収を0から40まで100点並べた表を作る。

In [ ]:
grid64 = pd.DataFrame({'age': 64, 'income': np.linspace(0, 40, 100)})
grid64.head(3).round(2)

3つの確率を描く。`ax.axhline(1, color='gray', linewidth=0.8)` は、確率1の位置に灰色の細い横線を引く。

In [ ]:
fig, ax = plt.subplots()
ax.plot(grid64['income'], stats.norm.cdf(-2.3 + 0.04 * grid64['age'] + 0.09 * grid64['income']),
        label='真の確率')
ax.plot(grid64['income'], lpm.predict(grid64), label='線形確率モデル')
ax.plot(grid64['income'], probit.predict(grid64), label='プロビット')
ax.axhline(1, color='gray', linewidth=0.8)
ax.set(xlabel='年収（百万円）', ylabel='持ち家の確率')
ax.legend()
plt.show()

線形確率モデルの直線はデータの範囲の中の年収13百万円ほどで1を超える。プロビットの曲線は1に近づくが、1を超えない。プロビットの曲線は真の確率の曲線とも近い。

## 6. 結果の読み方

プロビットは1次式を $\Phi$ に入れて確率を表すので、予測確率が0と1の範囲を外れない。1次式の値が近い世帯の持ち家の割合は $\Phi$ の曲線に沿い、潜在変数 $y^*=z+\varepsilon$ の $\varepsilon$ が標準正規分布に従うと考えると、確率を表す曲線は $\Phi$ になる。係数は観測されたデータが最も起こりやすくなる係数を選ぶ最尤法で推定する。このデータでは推定値は真の係数に近く、64歳で年収20百万円の世帯の予測確率は0.978だった。

データの世帯の多くは $\Phi$ の曲線が直線に近い範囲にいるので、線形確率モデルの直線も真の確率をよく近似する。2つのモデルの違いは、64歳で年収20百万円の世帯のように、予測確率が1に近い世帯に表れる。

係数は括弧の中の1次式の変化を表し、その符号は確率が上がる向きを表す。年齢が1歳高いと確率がどれだけ上がるかは、第9回の限界効果で調べる。

## 参考文献

- [Heiss, F. and Brunner, D. (2024) Using Python for Introductory Econometrics, 2nd edition](https://www.urfie.net/downloads/PDF/UPfIE_web.pdf)、17.1.2節、17.1.4節
- Wooldridge, J. M. (2001) Econometric Analysis of Cross Section and Panel Data, MIT Press、15.3節

## 関数の資料

- [pandas：数値を区分に分ける（`cut`）](https://pandas.pydata.org/docs/reference/api/pandas.cut.html)
- [SciPy：標準正規分布（`stats.norm`）](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.norm.html)
- [statsmodels：式によるプロビット（`smf.probit`）](https://www.statsmodels.org/stable/api.html#statsmodels-formula-api)
- [statsmodels：結果の予測（`predict`）](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.predict.html)